# Offline evaluation on `shop.history_sessions`

Before putting a new recommender live, estimate offline whether it beats the baseline.

The log was produced by the **old system** (best sellers plus a bit of randomness), so it only tells you what happened with the products *it* showed. Two estimates:

1. **Hit rate**: of the impressions where the shopper clicked, how often is the clicked product in your top 10? Easy, but biased towards popular products (the old system showed them most).
2. **Inverse propensity scoring (IPS)** of the click rate in **slot 1**: keep only the impressions where your first product equals the one the old system put in slot 1, and weight each by `1 / propensity`. This corrects the popularity bias of the log for that slot. SNIPS (self-normalised IPS) has lower variance.

We hold out the **last 7 days** of the history: the policies are built only from data before the cut and evaluated after it.

In [1]:
import os
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import requests

for line in Path("../.env").read_text().splitlines() if Path("../.env").exists() else []:
    key, _, value = line.partition("=")
    os.environ.setdefault(key.strip(), value.strip())
API = os.environ.get("RAWTREE_API_URL", "https://api.rawtree.com")
http = requests.Session()
http.headers["Authorization"] = f"Bearer {os.environ['RAWTREE_API_KEY']}"

def sql(q):
    r = http.post(f"{API}/v1/query", json={"sql": q}, timeout=600)
    r.raise_for_status()
    return pd.DataFrame(r.json()["data"])

## Data

In [2]:
last_day = sql("SELECT toDate(max(ts::DateTime64(3))) AS d FROM shop.history_sessions").d[0]
CUT = str((pd.Timestamp(last_day) - pd.Timedelta(days=7)).date())   # 'YYYY-MM-DD'
print("train < ", CUT, "<= test")

# Columns are Dynamic (schemaless): filters work as is; joins and some aggregates need casts like ::String
test = sql(f"""
    SELECT request_id, ts, user_id, context, recent_views, shown, propensity, clicked, purchased
    FROM shop.history_sessions WHERE ts >= '{CUT}' ORDER BY rand() LIMIT 100000""")
test["clicked"] = test.clicked.replace("", None)
print(len(test), "test impressions, observed CTR", round(test.clicked.notna().mean(), 4))
test.head(3)

train <  2026-11-07 <= test


100000 test impressions, observed CTR 0.0377


,request_id,ts,user_id,context,recent_views,shown,propensity,clicked,purchased
0,r_ab70277e7d20c824,2026-11-08 19:18:46.115000000,u_53479dd0,"{'device': 'android', 'page': 'category', 'reg...","[{'product_id': 'p_40e27a', 'ts': '2026-11-08 ...","[p_f487a6, p_ba0231, p_0d6a41, p_704766, p_a52...","[8.2e-05, 0.015671, 0.001157, 0.000405, 0.0360...",None,None
1,r_4bea90cabd64b454,2026-11-11 17:32:47.663000000,u_8318611e,"{'device': 'ios', 'page': 'product', 'region':...","[{'product_id': 'p_e0affc', 'ts': '2026-11-11 ...","[p_7c96ce, p_5cdf91, p_3d987d, p_f2cbdd, p_20d...","[0.029312, 0.015131, 0.223427, 0.012337, 0.006...",None,None
2,r_285739ea5763ef55,2026-11-11 19:43:49.442000000,u_87776405,"{'device': 'android', 'page': 'category', 'reg...","[{'product_id': 'p_093dac', 'ts': '2026-11-11 ...","[p_e39e79, p_ed457e, p_3033ed, p_1bcbe1, p_7c9...","[0.011451, 0.012902, 0.002187, 0.173209, 0.009...",None,None


In [3]:
import sys
sys.path.insert(0, ".")
from recommender import best_sellers, recommend   # the same code that runs live

best = best_sellers(until=CUT)

predictor ready: lgbm-v1 · 5,100 products (5,063 active)


In [4]:
# Co-views computed inside RawTree: for every product, the 20 products most viewed in the same sessions
coviews = sql(f"""
    SELECT a.product_id AS p, b.product_id AS q, count() AS n
    FROM (SELECT session_id::String AS session_id, product_id::String AS product_id FROM shop.history_events WHERE event_type = 'view' AND ts < '{CUT}') AS a
    INNER JOIN (SELECT session_id::String AS session_id, product_id::String AS product_id FROM shop.history_events WHERE event_type = 'view' AND ts < '{CUT}') AS b
      ON a.session_id = b.session_id
    WHERE a.product_id != b.product_id
    GROUP BY p, q ORDER BY n DESC LIMIT 20 BY p""")
covisit = coviews.groupby("p").q.apply(list).to_dict()
print(len(covisit), "products with co-views")

5000 products with co-views


## Policies

A policy maps one request (a row) to an ordered list of 10 product IDs.

In [5]:
def baseline(row):
    """The live baseline, imported from starter/recommender.py."""
    return recommend(row.recent_views, best)

def coview(row):
    """Products co-viewed with this session's views (most recent first), then the baseline."""
    score = Counter()
    for k, v in enumerate(reversed(row.recent_views)):
        for rank, q in enumerate(covisit.get(v["product_id"], [])):
            score[q] += 0.7 ** k / (rank + 1)
    items = [q for q, _ in score.most_common(10)]
    return (items + [p for p in baseline(row) if p not in items])[:10]

POLICIES = {"baseline": baseline, "coview": coview}

## Evaluation

In [6]:
def evaluate(policy, df):
    recs = [policy(r) for r in df.itertuples()]
    clicked = df.clicked.to_numpy()
    has_click = df.clicked.notna().to_numpy()
    hit = np.array([c in r for c, r in zip(clicked, recs)])[has_click].mean()
    first_shown = df.shown.str[0].to_numpy()
    match = np.array([r[0] == s for r, s in zip(recs, first_shown)])
    reward = (clicked == first_shown).astype(float)            # click on the slot-1 product
    w = match / df.propensity.str[0].to_numpy()
    return {"hit_rate@10": hit, "IPS_ctr_slot1": (w * reward).mean(),
            "SNIPS_ctr_slot1": (w * reward).sum() / max(w.sum(), 1e-9), "slot1_matches": int(match.sum())}

old_slot1 = (test.clicked == test.shown.str[0]).mean()
print(f"old system, observed slot-1 CTR: {old_slot1:.4f}")
pd.DataFrame({name: evaluate(p, test) for name, p in POLICIES.items()}).T

old system, observed slot-1 CTR: 0.0109


,hit_rate@10,IPS_ctr_slot1,SNIPS_ctr_slot1,slot1_matches
baseline,0.391938,0.007408,0.049800,3996.0
coview,0.229647,0.009341,0.013911,5451.0


## Our model (LightGBM) vs the policies above

`model/build_training.py` trains a gradient-boosted ranker on `shop.history_sessions`
(all click/purchase sessions + 7% negatives, weight = 1/propensity, features from
`features/build_features.py`). Here we score it against the same sessions the
heuristic policies get.

The model is slow per request (~30-40 ms), so we evaluate on a random **sample** of
the test set (all three policies on the *same* rows, apples-to-apples). Raise
`SAMPLE` if you are patient.

Caveats to keep in mind:
- the model was trained with a 10-day cut, the notebook holds out 7 days, and the
  feature tables (`features/out/`) were built on the full history — so the estimates
  below are slightly optimistic for the model. Retrain with the same CUT for a
  strictly clean comparison.
- hit-rate and IPS here measure *clicks*; the live score optimises *purchases*.

In [7]:
import sys, time
sys.path.insert(0, str(Path("..").resolve()))
from model.predictor import Predictor

pred = Predictor()
model_version = pred.VERSION
print("loaded", model_version)

def model_policy(row):
    req = {"request_id": row.request_id, "ts": str(row.ts), "user_id": row.user_id,
           "context": dict(row.context), "recent_views": list(row.recent_views)}
    try:
        return pred.recommend(req, unavailable=set())
    except Exception:
        return baseline(row)          # same safety net the live loop uses

SAMPLE = 1500
test_s = test.sample(min(SAMPLE, len(test)), random_state=0)
t0 = time.time()
model_eval = {name: evaluate(p, test_s) for name, p in POLICIES.items()} | {"model": evaluate(model_policy, test_s)}
print(f"evaluated {len(test_s):,} sampled sessions in {time.time()-t0:.0f}s")
pd.DataFrame(model_eval).T


predictor ready: lgbm-v1 · 5,100 products (5,063 active)
loaded lgbm-v1


evaluated 1,500 sampled sessions in 45s


,hit_rate@10,IPS_ctr_slot1,SNIPS_ctr_slot1,slot1_matches
baseline,0.454545,0.012408,0.087719,57.0
coview,0.218182,0.014621,0.024856,75.0
model,0.600000,0.007191,0.024932,74.0


## Next steps

- Few `slot1_matches` means a noisy IPS estimate: report it next to the estimate.
- Personal taste: who bought what in `shop.history_events` (matrix factorisation, item-to-item similarity).
- Cold start: new products only have a `description`. Text embeddings can place them next to similar products.
- Context: does price sensitivity change with `device` and `region`?
- Live: the stream changes (trends, price drops, stock-outs). Your `team_NN.feedback` tells you how you are really doing.